#### DBSCAN

In [39]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    adjusted_rand_score,
    normalized_mutual_info_score
)
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [40]:
data = pd.read_csv("../../../data/preprocessed/preprocessed_dataset.csv")
y = data["expression"].astype("category").cat.codes
class_names = data["expression"].astype("category").cat.categories

datasets = {
    "Scaled dataset": "../../../data/reduced/X_scaled.npy",
    "PCA dataset": "../../../data/reduced/X_pca.npy"
}

In [41]:
from sklearn.neighbors import NearestNeighbors

# k-distance graph does not show a clear elbow, indicating that 
# density-based clustering may not be well suited for this dataset.
def plot_k_distance(X, min_samples=10, dataset_name="dataset"):

    sample_size = 5000
    idx = np.random.choice(len(X), sample_size, replace=False)
    X_sample = X[idx]

    neighbors = NearestNeighbors(n_neighbors=min_samples)
    neighbors_fit = neighbors.fit(X_sample)

    distances, _ = neighbors_fit.kneighbors(X_sample)

    k_distances = np.sort(distances[:, min_samples-1])

    plt.figure(figsize=(6,4))
    plt.plot(k_distances)
    plt.title(f"k-distance graph ({dataset_name})")
    plt.xlabel("Points sorted by distance")
    plt.ylabel(f"{min_samples}-NN distance")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"../../../results/dbscan/dbscan_k_distance_{dataset_name}.png")
    plt.close()
    # plt.show()

In [42]:
def plot_knn_distance_distribution(X, min_samples=10, dataset_name="dataset", sample_size=5000, random_state=42):
    n_samples = len(X)

    if n_samples > sample_size:
        rng = np.random.default_rng(random_state)
        idx = rng.choice(n_samples, size=sample_size, replace=False)
        X_sample = X[idx]
    else:
        X_sample = X

    neighbors = NearestNeighbors(n_neighbors=min_samples)
    neighbors.fit(X_sample)

    distances, _ = neighbors.kneighbors(X_sample)

    kth_distances = distances[:, min_samples - 1]

    plt.figure(figsize=(7, 4))
    plt.hist(kth_distances, bins=50)
    plt.title(f"Distribution of {min_samples}-NN distances ({dataset_name})")
    plt.xlabel(f"Distance to {min_samples}-th nearest neighbor")
    plt.ylabel("Number of samples")
    plt.tight_layout()

    safe_name = dataset_name.replace(" ", "_").lower()
    plt.savefig(f"../../../results/dbscan/dbscan_knn_distance_distribution_{safe_name}.png")
    plt.close()

In [43]:
# Visualization shows a large number of small clusters and many noise points, 
# indicating unstable density regions.
def plot_tsne_dbscan(X, labels, dataset_name, sample_size=5000, random_state=42):
    n_samples = len(X)

    # Ako je dataset vec manji od sample_size, koristi ceo dataset
    if n_samples > sample_size:
        rng = np.random.default_rng(random_state)
        idx = rng.choice(n_samples, size=sample_size, replace=False)

        X_sample = X[idx]
        labels_sample = np.array(labels)[idx]
    else:
        X_sample = X
        labels_sample = np.array(labels)

    tsne = TSNE(n_components=2, perplexity=30, random_state=random_state)
    X_tsne = tsne.fit_transform(X_sample)

    plt.figure(figsize=(6,5))

    plt.scatter(
        X_tsne[:,0],
        X_tsne[:,1],
        c=labels_sample,
        cmap="tab20",
        s=6
    )

    plt.title(f"DBSCAN clusters ({dataset_name})")
    plt.xlabel("TSNE 1")
    plt.ylabel("TSNE 2")

    plt.tight_layout()
    plt.savefig(f"../../../results/dbscan/dbscan_tsne_{dataset_name}.png")
    plt.close()
    # plt.show()

In [44]:
import seaborn as sns

# The contingency matrix shows that most data points are assigned to the noise cluster (-1), 
# while the remaining clusters contain only small subsets of multiple classes.
def plot_contingency_heatmap(y_true, labels, class_names):

    df = pd.DataFrame({
        "True": y_true,
        "Cluster": labels
    })

    matrix = pd.crosstab(df["True"], df["Cluster"])
    matrix.index = class_names
    matrix = matrix.loc[:, matrix.sum() > 50]

    plt.figure(figsize=(12,6))

    sns.heatmap(
        matrix,
        cmap="Blues",
        cbar=True
    )

    plt.title("Contingency matrix (True classes vs DBSCAN clusters)")
    plt.xlabel("Cluster label")
    plt.ylabel("True class")

    plt.tight_layout()
    plt.savefig(f"../../../results/dbscan/dbscan_contingency_heatmap.png")
    plt.close()
    # plt.show()

In [45]:
# Most clusters contain very few samples, confirming that DBSCAN produced many micro-clusters 
# instead of meaningful groups.
def plot_cluster_size_distribution(labels):

    cluster_sizes = pd.Series(labels)
    cluster_sizes = cluster_sizes[cluster_sizes != -1].value_counts()

    plt.figure(figsize=(8,4))

    cluster_sizes.plot(kind="bar")

    plt.title("Cluster size distribution")
    plt.xlabel("Cluster label")
    plt.ylabel("Number of samples")

    plt.tight_layout()
    plt.savefig(f"../../../results/dbscan/dbscan_cluster_size_distribution.png")
    plt.close()
    # plt.show()

In [50]:
eps_values = [1.2, 1.5, 2.0, 2.5, 3.0]
min_samples = 30

results = []

for dataset_name, X_data in datasets.items():
    print("=" * 80)
    print(f"DATASET: {dataset_name}")
    print("=" * 80)
    X = np.load(X_data)

    plot_k_distance(X, min_samples=min_samples, dataset_name=dataset_name)
    plot_knn_distance_distribution(X, min_samples=min_samples, dataset_name=dataset_name)

    best_labels = None
    best_eps = None
    best_ari = -1
    best_nmi = -1

    for eps in eps_values:
        dbscan = DBSCAN(eps=eps, min_samples=min_samples, metric="euclidean", algorithm="ball_tree", n_jobs=-1)
        labels = dbscan.fit_predict(X)

        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = np.sum(labels == -1)
        noise_ratio = n_noise / len(labels)

        non_noise_mask = labels != -1
        unique_non_noise = np.unique(labels[non_noise_mask])

        if len(unique_non_noise) >= 2:
            silhouette = silhouette_score(
                X[non_noise_mask],
                labels[non_noise_mask],
                # sample_size=min(3000, np.sum(non_noise_mask))
                sample_size=2000
            )
            davies_bouldin = davies_bouldin_score(
                X[non_noise_mask],
                labels[non_noise_mask]
            )
        else:
            silhouette = np.nan
            davies_bouldin = np.nan

        ari = adjusted_rand_score(y, labels)
        nmi = normalized_mutual_info_score(y, labels)

        # biramo najbolji model po ARI
        if ari > best_ari:
            best_ari = ari
            best_nmi = nmi
            best_eps = eps
            best_labels = labels.copy()

    results.append({
        "Dataset": dataset_name,
        "Algorithm": "DBSCAN",
        "eps": best_eps,
        "min_samples": min_samples,
        "Clusters": n_clusters,
        "Noise_points": int(n_noise),
        "Noise_ratio": noise_ratio,
        "Silhouette": silhouette,
        "Davies_Bouldin": davies_bouldin,
        "ARI": best_ari,
        "NMI": best_nmi
    })

    print("Najbolji rezultat za dataset:", dataset_name)
    print(f"Best eps: {best_eps}")
    print(f"Best ARI: {best_ari:.4f}")
    print(f"Best NMI: {best_nmi:.4f}\n")

    # 3) Grafici samo za najbolji eps
    plot_tsne_dbscan(X, best_labels, dataset_name)
    plot_contingency_heatmap(y, best_labels, class_names)
    plot_cluster_size_distribution(best_labels)

DATASET: Scaled dataset
Najbolji rezultat za dataset: Scaled dataset
Best eps: 3.0
Best ARI: 0.0243
Best NMI: 0.1068

DATASET: PCA dataset
Najbolji rezultat za dataset: PCA dataset
Best eps: 3.0
Best ARI: 0.0267
Best NMI: 0.0938



In [51]:
results_df = pd.DataFrame(results)
results_df = results_df.round({
    "Noise_ratio": 4,
    "Silhouette": 4,
    "Davies_Bouldin": 4,
    "ARI": 4,
    "NMI": 4
})

results_df.to_csv("../../../results/dbscan/dbscan_grid_summary.csv", index=False)
print(results_df.sort_values(["Dataset", "ARI"], ascending=[True, False]))
print(results_df)

          Dataset Algorithm  eps  min_samples  Clusters  Noise_points  \
1     PCA dataset    DBSCAN  3.0           30        10         12468   
0  Scaled dataset    DBSCAN  3.0           30         8         14408   

   Noise_ratio  Silhouette  Davies_Bouldin     ARI     NMI  
1       0.4463      0.1091          0.9490  0.0267  0.0938  
0       0.5158      0.1652          1.0479  0.0243  0.1068  
          Dataset Algorithm  eps  min_samples  Clusters  Noise_points  \
0  Scaled dataset    DBSCAN  3.0           30         8         14408   
1     PCA dataset    DBSCAN  3.0           30        10         12468   

   Noise_ratio  Silhouette  Davies_Bouldin     ARI     NMI  
0       0.5158      0.1652          1.0479  0.0243  0.1068  
1       0.4463      0.1091          0.9490  0.0267  0.0938  


##### Primena DBSCAN algoritma na oba razmatrana skupa podataka (scaled dataset i PCA dataset) nije dovela do klastera koji odgovaraju stvarnoj strukturi klasa. Najbolji rezultati su postignuti za parametre eps = 3.0 i min_samples = 30, pri čemu je broj pronađenih klastera bio relativno mali (8 za scaled dataset i 10 za PCA dataset), ali je istovremeno identifikovan veliki broj tačaka označenih kao šum (oko 45–52% svih uzoraka).

##### Vrednosti eksterne evaluacije klasterovanja su veoma niske (ARI ≈ 0.02–0.03 i NMI ≈ 0.09–0.11), što ukazuje da pronađeni klasteri ne odgovaraju stvarnim klasama u datasetu. Takođe, unutrašnje metrike kvaliteta klasterovanja (Silhouette i Davies–Bouldin indeks) ne pokazuju jasnu separaciju između klastera.

##### Vizuelizacije u TSNE prostoru, kao i analiza distribucije veličina klastera, pokazuju da DBSCAN formira nekoliko većih klastera uz značajan broj izolovanih tačaka i manjih klastera. To ukazuje da dataset ne poseduje jasno izraženu gustinsku strukturu koja bi bila pogodna za algoritme zasnovane na gustini.

##### Na osnovu ovih rezultata može se zaključiti da DBSCAN nije pogodan algoritam za klasterovanje ovog skupa podataka, verovatno zbog visoke dimenzionalnosti i preklapanja distribucija između klasa, što otežava identifikaciju stabilnih gustinskih regiona u prostoru osobina.

In [54]:
X = np.load(datasets["PCA dataset"])

sample_size = 5000
np.random.seed(42)
idx = np.random.choice(len(X), sample_size, replace=False)
X_sample = X[idx]
y_sample = y[idx]

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X_sample)

plt.figure(figsize=(6,5))
plt.scatter(
    X_tsne[:,0],
    X_tsne[:,1],
    c=y_sample,
    cmap="tab20",
    s=6
)
plt.title(f"True class TSNE (PCA dataset)")
plt.xlabel("TSNE 1")
plt.ylabel("TSNE 2")
plt.tight_layout()
plt.savefig(f"../../../results/dbscan/dbscan_tsne_true_classes.png")
plt.close()